# Elasticsearch Ingest

This notebook loads the FAQ documents and indexes them into Elasticsearch instead of SQLite.

In [1]:
%pip install elasticsearch

/workspaces/rag_app/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1342 documents


In [3]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 79 documents


In [ ]:
import os
import time
from elasticsearch import Elasticsearch

ELASTIC_URL = os.getenv("ELASTIC_URL", "http://localhost:9200")
INDEX_NAME = os.getenv("ELASTIC_INDEX_NAME", "faq")

client = Elasticsearch(ELASTIC_URL)

if not client.ping():
    raise RuntimeError(f"Elasticsearch is not reachable at {ELASTIC_URL}. Start Elasticsearch or set ELASTIC_URL to a running cluster.")

index_config = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "question": {"type": "text"},
            "section": {"type": "text"},
            "answer": {"type": "text"},
            "course": {"type": "keyword"},
        }
    },
}

if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)

client.indices.create(index=INDEX_NAME, **index_config)

for i, doc in enumerate(docs_llm):
    client.index(index=INDEX_NAME, id=i, document=doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.1)

client.indices.refresh(index=INDEX_NAME)
print(f"Done. Documents indexed in Elasticsearch index '{INDEX_NAME}'")

ConnectionError: Connection error caused by: ConnectionError(Connection error caused by: NewConnectionError(HTTPConnection(host='localhost', port=9200): Failed to establish a new connection: [Errno 111] Connection refused))